# 0. Library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# 1. Function definitions

In [2]:
def f_monthly(
        mortgage_total, 
        annual_repayment_rate, 
        annual_interest_rate, 
        months, 
        monthly_avgift=0,
        plot=True,
        title='Monthly payment & balance',
        show_payment=True,
        show_balance=True
        ):
    """Return per-month lists (length `months`) instead of only the last month.

    The repayment is fixed each month, computed from the original mortgage amount;
    interest is still charged on the outstanding balance.
    """
    monthly_interest_rate = annual_interest_rate/12
    monthly_repayment_rate = annual_repayment_rate/12
    fixed_repayment = mortgage_total * monthly_repayment_rate  # based on original amount
    mortgage_paid = 0
    paid, remain, repayment, interest, avgift, total = [], [], [], [], [], []
    
    for m in np.arange(months):
        mortgage_remain = mortgage_total - mortgage_paid
        monthly_repayment = min(fixed_repayment, mortgage_remain)
        monthly_interest = mortgage_remain * monthly_interest_rate
        monthly_total = monthly_repayment + monthly_interest + monthly_avgift
        mortgage_paid += monthly_repayment
        paid.append(mortgage_paid)                    # cumulative, end of month
        remain.append(mortgage_total - mortgage_paid) # end of month
        repayment.append(monthly_repayment)
        interest.append(monthly_interest)
        avgift.append(monthly_avgift)
        total.append(monthly_total)
    
    if plot:
        months_axis = np.arange(1, months+1)
        hover = '%{y:,.2~f}<extra></extra>'

        fig = go.Figure()
        layout = dict(
            title=title, xaxis_title='month', hovermode='x unified',
            legend=dict(groupclick='toggleitem', x=1.12, xanchor='left', y=1),
            margin=dict(l=60, r=160, t=40, b=50)
            )

        if show_payment:
            fig.add_trace(go.Scatter(
                x=months_axis, y=repayment, name='repayment', hovertemplate=hover,
                legendgroup='payment', legendgrouptitle_text='Left axis (payment)'
                ))
            fig.add_trace(go.Scatter(x=months_axis, y=interest, name='interest', hovertemplate=hover, legendgroup='payment'))
            fig.add_trace(go.Scatter(x=months_axis, y=avgift, name='avgift', hovertemplate=hover, legendgroup='payment'))
            fig.add_trace(go.Scatter(x=months_axis, y=total, name='total', hovertemplate=hover, legendgroup='payment'))
            layout['yaxis'] = dict(title='SEK / month')

        if show_balance:
            fig.add_trace(go.Scatter(
                x=months_axis, y=paid, name='paid', yaxis='y2', line=dict(dash='dash'), hovertemplate=hover,
                legendgroup='balance', legendgrouptitle_text='Right axis (balance)'
                ))
            fig.add_trace(go.Scatter(
                x=months_axis, y=remain, name='remaining', yaxis='y2', line=dict(dash='dash'), hovertemplate=hover,
                legendgroup='balance'
                ))
            layout['yaxis2'] = dict(title='SEK balance', overlaying='y', side='right')

        fig.update_layout(**layout)
        fig.show()

    return paid, remain, repayment, interest, avgift, total

def f_compound(
        base,
        year_plan,
        periods_per_year=12,
        plot=False,
        title='Cumulative savings'
        ):
    """Return per-period lists of cumulative savings (period = 1/periods_per_year of a
    year; default monthly). Both the simulation length and the growth rate are
    derived from `year_plan` rather than passed separately.

    Each period, interest is credited on the current balance, then that period's net
    addition/subtraction is applied. `yearly_growth_rate` and `yearly_addition` are
    annual figures that get translated to their period equivalent:
      - the rate is converted to the compounding-equivalent per-period rate, so
        `periods_per_year` periods at that rate grow the balance by exactly the
        annual rate;
      - a recurring addition is split evenly across the periods it covers, and
        `yearly_addition_change` compounds continuously (fractional exponent of
        elapsed time) rather than jumping once per calendar year, so a segment
        that starts or ends mid-year is handled smoothly.

    `year_plan` is a list of segment dicts describing additions or subtractions over
    an inclusive year range. Years may be fractional (e.g. 1.5, 2.1):
        {
            'year': [start, end],            # inclusive year range, e.g. [1, 5.5];
                                               # a single year may be given as 10 or
                                               # [10] instead of [10, 10] -- a
                                               # single-value entry is a one-time
                                               # event (e.g. a lump sum), applied in
                                               # full at that point in time rather
                                               # than spread across periods
            'yearly_addition': 12000,         # amount added each year in range (or,
                                               # for a one-time entry, the full lump
                                               # amount); negative = withdrawal.
                                               # Default 0.
            'yearly_addition_change': 0.02,   # fractional growth of yearly_addition
                                               # per elapsed year within the range.
                                               # Default 0. Ignored for one-time
                                               # entries.
            'yearly_growth_rate': 0.09,       # growth rate for years in this range.
                                               # Optional: carried forward from the
                                               # last year it was set (0 before any
                                               # segment sets it).
        }
    `years` (the length of the simulation) is the highest year covered by any
    segment. Years not covered by any segment get no addition. Segments may
    overlap: their additions are summed (so a lump sum is just a one-year-range
    segment layered on a longer recurring plan); if segments disagree on the growth
    rate for an overlapping period, the last one in the list wins.
    """
    if not year_plan:
        raise ValueError('year_plan must contain at least one segment')

    normalized = []
    max_year = 0
    for segment in year_plan:
        segment_year = segment['year']
        if isinstance(segment_year, (list, tuple)):
            y_start, y_end = segment_year if len(segment_year) == 2 else (segment_year[0], segment_year[0])
        else:
            y_start = y_end = segment_year
        normalized.append((y_start, y_end, segment))
        max_year = max(max_year, y_end)

    periods = int(round(max_year * periods_per_year))
    additions = np.zeros(periods)
    rates = np.full(periods, np.nan)

    for y_start, y_end, segment in normalized:
        addition = segment.get('yearly_addition', 0)
        addition_change = segment.get('yearly_addition_change', 0)
        rate_override = segment.get('yearly_growth_rate')

        if y_start == y_end:
            # one-time event (e.g. lump sum): applied in full at the single period
            # nearest to that point in time, not spread across periods_per_year
            idx = int(round(y_end * periods_per_year)) - 1
            idx = min(max(idx, 0), periods - 1)
            additions[idx] += addition
            if rate_override is not None:
                rates[idx] = rate_override
        else:
            start_idx = max(int(round(max(y_start - 1, 0) * periods_per_year)), 0)
            end_idx = min(int(round(y_end * periods_per_year)), periods)

            for offset in range(end_idx - start_idx):
                idx = start_idx + offset
                elapsed_years = offset / periods_per_year
                period_addition = (addition / periods_per_year) * (1 + addition_change) ** elapsed_years
                additions[idx] += period_addition
                if rate_override is not None:
                    rates[idx] = rate_override

    last_annual_rate = 0.0
    period_rates = np.zeros(periods)
    for idx in range(periods):
        if not np.isnan(rates[idx]):
            last_annual_rate = rates[idx]
        period_rates[idx] = (1 + last_annual_rate) ** (1/periods_per_year) - 1

    balance = base
    contributed = base
    contributed_list, interest_list, addition_list, total = [], [], [], []

    for idx in np.arange(periods):
        period_interest = balance * period_rates[idx]
        balance += period_interest + additions[idx]
        contributed += additions[idx]
        contributed_list.append(contributed)          # cumulative, end of period
        interest_list.append(balance - contributed)   # cumulative, end of period
        addition_list.append(additions[idx])
        total.append(balance)                          # cumulative, end of period

    if plot:
        years_axis = np.arange(1, periods+1) / periods_per_year
        hover = '%{y:,.2~f}<extra></extra>'

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=years_axis, y=contributed_list, name='contributed', stackgroup='one', hovertemplate=hover))
        fig.add_trace(go.Scatter(x=years_axis, y=interest_list, name='interest', stackgroup='one', hovertemplate=hover))
        fig.add_trace(go.Scatter(x=years_axis, y=total, name='total', line=dict(dash='dot', color='black'), hovertemplate=hover))
        fig.update_layout(
            title=title, xaxis_title='year', yaxis_title='SEK', hovermode='x unified',
            margin=dict(l=60, r=20, t=40, b=50)
            )
        fig.show()

    return contributed_list, interest_list, addition_list, total

# 2. Housing payments estimator

In [3]:
AVGIFT = 4600+12000/12 # per month
MORTGAGE = 3.0e6-3.0e5
REPAYMENT_RATE = 0.02 # per year
INTEREST_RATE = 0.03 # per year
MONTHS = 12*30

paid, remain, monthly_repayment, monthly_interest, monthly_avgift, monthly_total = f_monthly(
    mortgage_total=MORTGAGE, 
    annual_repayment_rate=REPAYMENT_RATE,
    annual_interest_rate=INTEREST_RATE, 
    months=MONTHS, 
    monthly_avgift=AVGIFT, 
    plot=True,
    title='Monthly payment & balance (Brommaplan)'
    )

# 3. Cumulative savings estimator

In [7]:
BASE = 1.7e5
ADDITION_GROWTH_RATE = 0.02 # yearly addition increases % per year
YEARLY_RATE = 0.09 # per year

YEAR_PLAN = [
    # saving X / month
    {
        'year': [1, 10], 
        'yearly_addition': 12*20000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        }, 
    # saving X / month
    {
        'year': [11, 20], 
        'yearly_addition': 12*25000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        },
    # lump sum withdrawal
    {
        'year': 1.5, 
        'yearly_addition': -3e5
        },
    # lump sum withdrawal
    {
        'year': 10, 
        'yearly_addition': -0.5e6
        },
    # lump sum withdrawal
    {
        'year': 15, 
        'yearly_addition': -1e6
        },
    # no addition, subtraction only
    {
        'year': [21, 35], 
        'yearly_addition': -12*30000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
    },
    # # no addition, no subtraction
    # {
    #     'year': [36, 40], 
    #     'yearly_addition': 0, 
    #     'yearly_addition_change': 0, 
    #     'yearly_growth_rate': YEARLY_RATE
    # },
    # lump sum withdrawal
    {
        'year': 30, 
        'yearly_addition': -4e6
        },
    # no addition, no subtraction
    {
        'year': [36, 50], 
        'yearly_addition': -12*10000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
    },
]

contributed, interest_earned, yearly_addition, total = f_compound(
    base=BASE,
    year_plan=YEAR_PLAN,
    plot=True,
    title='Cumulative savings (Y)'
    )

In [8]:
BASE = 4e4+2e4+8e3
ADDITION_GROWTH_RATE = 0.0 # yearly addition increases % per year
YEARLY_RATE = 0.09 # per year

YEAR_PLAN = [
    # saving X / month
    {
        'year': [1, 20], 
        'yearly_addition': 12*6000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        }, 
    # no addition, subtraction only
    {
        'year': [21, 55], 
        'yearly_addition': -12*20000, 
        'yearly_addition_change': 0.02, 
        'yearly_growth_rate': YEARLY_RATE
    },
    # lump sum withdrawal
    {
        'year': 1.5, 
        'yearly_addition': 3e5
        },
    # lump sum withdrawal
    {
        'year': 20, 
        'yearly_addition': -1.5e6
        },
    # no addition, no subtraction
    # {
    #     'year': [36, 40], 
    #     'yearly_addition': 0, 
    #     'yearly_addition_change': 0, 
    #     'yearly_growth_rate': YEARLY_RATE
    # },
]

contributed, interest_earned, yearly_addition, total = f_compound(
    base=BASE,
    year_plan=YEAR_PLAN,
    plot=True,
    title='Cumulative savings (A)'
    )